# КИМ 2.1. Backprop и обучение сети

**Модуль 2. Обучение искусственных нейронных сетей** · Курс «Основы нейронных сетей» · УрФУ

Проверяемый результат (КРМ v3.0):
- **DL-1.1 (Б):** реализует backprop, выбирает активацию, связность и
  инициализацию, сравнивает размеры пакета и применяет регуляризацию.

Подробное описание, критерии и контрольные вопросы — в `kim-01-backprop-training.md`.

---
## Часть А. Backprop на чистом NumPy (обязательно)

Цель — увидеть, как градиенты реально текут через вычислительный граф. Реализуем
двухслойную сеть $784 \to 64 \to 10$ и алгоритм обратного распространения ошибки
**вручную**, через правило цепи.

Архитектура:
$$z_1 = W_1 x + b_1, \quad a_1 = \mathrm{ReLU}(z_1), \quad z_2 = W_2 a_1 + b_2, \quad \hat{y} = \mathrm{softmax}(z_2)$$

Функция потерь — кросс-энтропия: $L = -\sum_k y_k \log \hat{y}_k$.

### 0. Импорт библиотек и загрузка подмножества Fashion-MNIST

In [ ]:
# < ENTER YOUR CODE HERE >

Загрузите Fashion-MNIST и возьмите подмножество (например, 10 000 примеров
для скорости): преобразуйте в вектор длины 784, нормируйте в $[0, 1]$.
`one-hot` для меток обязателен — будем использовать кросс-энтропию напрямую.

### 1. Инициализация параметров

Инициализируйте `W1` (784×64), `b1` (64), `W2` (64×10), `b2` (10) малыми
случайными значениями. Зафиксируйте `np.random.seed(42)`.

In [ ]:
# < ENTER YOUR CODE HERE >  # W1, b1, W2, b2

### 2. Прямой проход

Реализуйте `forward(X)` → `(z1, a1, z2, y_hat)`. Не забудьте `softmax` на выходе.

In [ ]:
# < ENTER YOUR CODE HERE >  # forward

### 3. Функция потерь

Реализуйте `loss(y_hat, y)` — кросс-энтропия. Добавьте малый $\varepsilon$ в
`log` во избежание $\log 0$.

In [ ]:
# < ENTER YOUR CODE HERE >  # loss

### 4. Обратный проход (backprop)

Реализуйте `backward(X, y, z1, a1, z2, y_hat)` → `(dW1, db1, dW2, db2)`.

Подсказка — производные через правило цепи (для softmax + cross-entropy):
$$\frac{\partial L}{\partial z_2} = \hat{y} - y, \quad
\frac{\partial L}{\partial W_2} = a_1^T (\hat{y} - y), \quad
\frac{\partial L}{\partial a_1} = (\hat{y} - y) W_2^T$$
$$\frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial a_1} \odot \mathbb{1}[z_1 > 0] \text{ (производная ReLU)}, \quad
\frac{\partial L}{\partial W_1} = X^T \frac{\partial L}{\partial z_1}$$

In [ ]:
# < ENTER YOUR CODE HERE >  # backward

### 5. Цикл обучения

Реализуйте цикл по эпохам (50–100), внутри — мини-батчи (например, по 64 примера):
прямой проход → loss → backward → обновление `W -= lr * dW`.

In [ ]:
# < ENTER YOUR CODE HERE >  # training loop, lr=0.1

### 6. Проверка: loss убывает, точность растёт

Посчитайте и выведите loss и accuracy на train после каждой эпохи. Постройте
графики. Убедитесь, что loss монотонно убывает, а точность растёт.

In [ ]:
# < ENTER YOUR CODE HERE >  # графики loss и accuracy

**Контрольный вопрос:** на чём основан ваш backprop? Сформулируйте
правило цепи и покажите, как оно применяется к $\partial L / \partial W_1$.

> *Ваш ответ:* ...

---
## Часть Б. Цикл обучения на фреймворке

### 7. Та же сеть на Keras (`fit`) или PyTorch (явный цикл)

Перепишите двухслойную сеть средствами фреймворка и обучите через `fit`
(для Keras) или явный цикл с `optimizer.zero_grad()`, `forward`, `loss.backward()`,
`optimizer.step()` (для PyTorch).

In [ ]:
# < ENTER YOUR CODE HERE >

### 8. Сравнение GD и SGD

Запустите обучение с разными `batch_size` (10, 50, 200, 500). Сравните, как
меняется скорость и устойчивость сходимости. Сделайте вывод.

In [ ]:
# < ENTER YOUR CODE HERE >  # batch_size эксперимент

### 9. Выбор активации, инициализации и связности

В одинаковом протоколе сравните минимум три конфигурации:
- один скрытый слой, `ReLU` и инициализация He;
- один скрытый слой, `tanh` и инициализация Xavier;
- два скрытых слоя, `ReLU` и инициализация He.

Зафиксируйте seed, число эпох, optimizer и batch size. Сведите validation loss,
accuracy, время и число параметров в таблицу. Выберите конфигурацию и обоснуйте
активацию, инициализацию и степень связности.

In [ ]:
# < ENTER YOUR CODE HERE >  # три конфигурации + таблица + обоснованный выбор

---
## Часть В. Переобучение и регуляризация

### 10. Обнаружение переобучения

Увеличьте число эпох до 100–200. Постройте на одном графике `loss`/`val_loss` и
`accuracy`/`val_accuracy`. Найдите эпоху, после которой `val_loss` начинает расти
(переобучение).

In [ ]:
# < ENTER YOUR CODE HERE >  # кривые train/val

### 11. Регуляризация

Примените хотя бы один метод регуляризации: слой `Dropout`, `L2`, или
`EarlyStopping`. Сравните кривые до и после — расхождение train/val должно
уменьшиться.

In [ ]:
# < ENTER YOUR CODE HERE >  # Dropout / L2 / EarlyStopping

### 12. Вывод

Какие методы регуляризации оказались эффективнее и почему? Запишите краткий вывод.

> *Ваш вывод:* ...

---
## Итог

Перед сдачей убедитесь, что:
- [ ] backprop на NumPy работает (loss убывает, точность растёт);
- [ ] сравнены активации, инициализации и один/два скрытых слоя;
- [ ] построены графики кривых обучения;
- [ ] найдено переобучение и применена регуляризация;
- [ ] вы готовы вывести градиенты по слоям на защите.

**Формат сдачи:** заполненный ноутбук + устная защита.